# Canonical master object (Part B)

Fixes the issues that block the Figure 2 expression panels:

1. `X` currently holds **raw integer UMI counts**, not normalised data. Every dot plot and feature plot drawn from it so far is showing raw counts.
2. Gene naming is inconsistent across objects — your data uses `APLNR`, the legacy reference uses `APLNR (ENSG00000134817)`, the Tyser `.raw` uses `APLNR`.
3. The `48h` timepoint is really **42h**.
4. Several byte-identical duplicate objects exist across `02_new_experiment` and `04_processed`.

Writes one new file: `04_processed/adata_master_normalized.h5ad`. Nothing is overwritten or deleted. Section 7 only *reports* duplicates — it does not remove them.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1

import sys
sys.path.insert(0, str(Path.cwd().parent / "utils"))   # run notebooks from their own folder
from config import ROOT, PROJECT_DIR, CELLRANGER_DIR

MASTER_IN  = ROOT / "00_DATA_FROZEN" / "02_annotated" / "adata_full_joint_OCM_labels_corrected.h5ad"
MASTER_OUT = ROOT / "04_processed" / "adata_master_normalized.h5ad"
TYSER_IN   = ROOT / "01_references" / "embryo_tyser" / "240925_adata_tyser_hvg.h5ad"
TYSER_OUT  = ROOT / "04_processed" / "tyser_embryo_genes_for_fig2.h5ad"
TAB_DIR    = ROOT / "06_tables" / "master_build"
TAB_DIR.mkdir(parents=True, exist_ok=True)

GENES_OF_INTEREST = ["APLNR", "APELA", "APLN"]

print("ROOT:", ROOT)
print("in  :", MASTER_IN.name, "->", "OK" if MASTER_IN.exists() else "MISSING")
print("out :", MASTER_OUT.name, "->", "already exists" if MASTER_OUT.exists() else "will be created")

## 1. Load and confirm `X` is raw counts

Guard against double-normalising. If this object has already been processed, the assertions below will stop you.

In [ ]:
adata = ad.read_h5ad(MASTER_IN)
print("loaded:", adata.shape)

X = adata.X
sample = X.data[:200_000] if sp.issparse(X) else np.asarray(X[:200]).ravel()

is_integer = bool(np.all(sample == np.round(sample)))
xmax = float(sample.max())
already_logged = "log1p" in adata.uns

print(f"X integer-valued: {is_integer}")
print(f"X sample max    : {xmax:.2f}")
print(f"uns has 'log1p' : {already_logged}")

if "counts" in adata.layers:
    c = adata.layers["counts"]
    csample = c.data[:200_000] if sp.issparse(c) else np.asarray(c[:200]).ravel()
    print(f"X identical to layers['counts']: {np.array_equal(sample, csample)}")

if already_logged or not is_integer:
    raise RuntimeError(
        "X does not look like raw counts (already normalised?). "
        "Stopping so this notebook cannot double-normalise."
    )
print("\nConfirmed: X is raw counts. Safe to proceed.")

## 2. Normalise

Counts are preserved in `layers['counts']`. `X` becomes library-size normalised (1e4) and log1p-transformed — the scale scanpy's plotting functions assume when `use_raw=False`.

In [ ]:
# Preserve raw counts before touching X
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

adata.uns["normalization"] = {
    "method": "normalize_total(target_sum=1e4) + log1p",
    "raw_counts_in": "layers['counts']",
    "source_file": MASTER_IN.name,
}

newsample = adata.X.data[:200_000] if sp.issparse(adata.X) else np.asarray(adata.X[:200]).ravel()
print(f"X after normalisation: min={newsample.min():.3f} max={newsample.max():.3f}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm  : {list(adata.obsm.keys())}")

## 3. Harmonise gene naming

Three conventions are in play across your objects:

| object | convention | example |
|---|---|---|
| your master | plain symbol | `APLNR` |
| Tyser `var` (HVG) | symbol + Ensembl | `APLNR (ENSG00000134817)` |
| Tyser `.raw` | plain symbol | `APLNR` |
| legacy combined `.raw` | symbol + Ensembl | `APLNR (ENSG00000248329)` |

Everything is canonicalised to **plain symbol** in `var['symbol']`, with the reference-style name kept in `var['ref_style_name']` so you can join either direction.

In [ ]:
def to_symbol(names):
    """'APLNR (ENSG00000134817)' -> 'APLNR'; 'APLNR' -> 'APLNR'."""
    return pd.Index([str(n).split(" (")[0].strip() for n in names])

adata.var["symbol"] = to_symbol(adata.var_names)

if "gene_ids" in adata.var.columns:
    adata.var["ref_style_name"] = [
        f"{s} ({g})" for s, g in zip(adata.var["symbol"], adata.var["gene_ids"])
    ]
else:
    adata.var["ref_style_name"] = adata.var["symbol"]
    print("WARNING: no gene_ids column; ref_style_name falls back to symbol")

dupes = adata.var["symbol"].duplicated().sum()
print(f"duplicate symbols: {dupes}")
if dupes:
    print("  (kept as-is; use var['ref_style_name'] to disambiguate)")

print("\nGenes of interest in the master object:")
for g in GENES_OF_INTEREST:
    hit = adata.var.index[adata.var["symbol"] == g].tolist()
    print(f"  {g:8s} {'FOUND ' + str(hit) if hit else 'ABSENT'}")

## 4. Correct the timepoint label

The `48h` samples are actually **42h**. Original values are kept in `timepoint_original` so nothing is lost.

In [ ]:
TIME_FIX = {"48h": "42h", "72h": "72h"}

adata.obs["timepoint_original"] = adata.obs["timepoint"].astype(str)
adata.obs["timepoint_true"] = (
    adata.obs["timepoint_original"].map(TIME_FIX).astype("category")
)

unmapped = adata.obs.loc[adata.obs["timepoint_true"].isna(), "timepoint_original"].unique()
if len(unmapped):
    raise ValueError(f"Unmapped timepoint values: {unmapped}. Extend TIME_FIX.")

adata.obs["condition_time_true"] = (
    adata.obs["condition"].astype(str) + "_" + adata.obs["timepoint_true"].astype(str)
).astype("category")

print(pd.crosstab(adata.obs["timepoint_original"], adata.obs["timepoint_true"]))
print()
print(adata.obs["condition_time_true"].value_counts().sort_index().to_string())

## 5. Sanity checks before writing

Confirms the biological replicate structure survived, and that APLNR/APELA are actually measurable after normalisation — these are the panels Figure 2 depends on.

In [ ]:
comp = (adata.obs.groupby(["site", "condition", "timepoint_true"], observed=True)
        .size().rename("n_cells").reset_index()
        .sort_values(["site", "condition", "timepoint_true"]))
display(comp)
comp.to_csv(TAB_DIR / "cells_per_replicate.csv", index=False)

# sequencing depth by site - relevant because MAGIC should be run per site
depth = (adata.obs.groupby("site", observed=True)["total_counts"]
         .agg(["count", "median", "mean"]).round(1))
display(depth)

print("\nExpression of genes of interest (normalised, log1p):")
rows = []
for g in GENES_OF_INTEREST:
    idx = adata.var.index[adata.var["symbol"] == g]
    if len(idx) == 0:
        rows.append({"gene": g, "status": "ABSENT"})
        continue
    v = adata[:, idx[0]].X
    v = np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v).ravel()
    rows.append({
        "gene": g, "status": "present",
        "pct_cells_expressing": round(float((v > 0).mean() * 100), 2),
        "mean_lognorm": round(float(v.mean()), 4),
        "max_lognorm": round(float(v.max()), 4),
    })
display(pd.DataFrame(rows))

## 6. Write the canonical object

Refuses to overwrite an existing file unless you explicitly set `OVERWRITE = True`.

In [ ]:
WRITE = True
OVERWRITE = False

print("About to write:")
print(f"  path   : {MASTER_OUT}")
print(f"  shape  : {adata.shape}")
print(f"  X      : normalised + log1p")
print(f"  layers : {list(adata.layers.keys())}")
print(f"  obsm   : {list(adata.obsm.keys())}")
print(f"  new obs: timepoint_true, timepoint_original, condition_time_true")
print(f"  new var: symbol, ref_style_name")

if not WRITE:
    print("\nDRY RUN - set WRITE = True to save.")
elif MASTER_OUT.exists() and not OVERWRITE:
    print(f"\nSKIPPED - {MASTER_OUT.name} already exists. Set OVERWRITE = True to replace.")
else:
    ad.settings.allow_write_nullable_strings = True
    adata.write_h5ad(MASTER_OUT)
    size_gb = MASTER_OUT.stat().st_size / 1e9
    print(f"\nWrote {MASTER_OUT.name} ({size_gb:.2f} GB)")

## 7. Duplicate audit (report only)

Compares candidate duplicates by size, then by SHA-256 of the first and last 8 MB (full hashing of multi-GB files is slow and the head+tail check is sufficient to confirm identity in practice).

**This cell deletes nothing.** It prints a list for you to act on manually.

In [ ]:
def quick_hash(path, chunk=8 * 1024 * 1024):
    size = path.stat().st_size
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        h.update(fh.read(chunk))
        if size > chunk:
            fh.seek(max(0, size - chunk))
            h.update(fh.read(chunk))
    h.update(str(size).encode())
    return h.hexdigest()

candidates = sorted(
    list((ROOT / "04_processed").glob("*.h5ad"))
    + list((ROOT / "02_new_experiment").rglob("*.h5ad"))
)
candidates = [p for p in candidates if p.resolve() != MASTER_OUT.resolve()]

by_size = {}
for p in candidates:
    by_size.setdefault(p.stat().st_size, []).append(p)

groups = []
for size, paths in by_size.items():
    if len(paths) < 2:
        continue
    by_hash = {}
    for p in paths:
        by_hash.setdefault(quick_hash(p), []).append(p)
    for hsh, plist in by_hash.items():
        if len(plist) > 1:
            groups.append((size, plist))

total_reclaimable = 0
if not groups:
    print("No duplicate groups found.")
for size, plist in sorted(groups, key=lambda t: -t[0]):
    reclaim = size * (len(plist) - 1)
    total_reclaimable += reclaim
    print(f"\n{size/1e6:,.0f} MB x {len(plist)} copies  (reclaimable {reclaim/1e9:.2f} GB)")
    for p in plist:
        print(f"    {p.relative_to(ROOT)}")

print(f"\nTotal reclaimable: {total_reclaimable/1e9:.2f} GB")
print("Nothing was deleted. Review the list and remove copies yourself.")

## 8. Embryo gene-expression object for the Figure 2 APLNR/APELA panel

The Tyser HVG `var` (1,338 genes) contains **APLNR but not APELA**. APELA exists only in `.raw` (21,080 genes). This builds a small object carrying the full `.raw` gene space alongside the reference's own `cluster_id` labels and UMAP — which is all panel 3 needs, and requires no mapping.

Note `adata_tyser_10_24_24.h5ad` has an empty `.raw` (0 genes) — this uses `240925_adata_tyser_hvg.h5ad` instead.

In [ ]:
tyser = ad.read_h5ad(TYSER_IN)
print("tyser HVG object:", tyser.shape)

if tyser.raw is None or tyser.raw.shape[1] == 0:
    raise ValueError("Tyser .raw is empty - cannot recover APELA.")

tyser_full = tyser.raw.to_adata()
tyser_full.obs = tyser.obs.copy()
for k in tyser.obsm.keys():
    tyser_full.obsm[k] = tyser.obsm[k]

tyser_full.var["symbol"] = to_symbol(tyser_full.var_names)
print("full gene space:", tyser_full.shape)

print("\nGenes of interest in Tyser .raw:")
for g in GENES_OF_INTEREST:
    hit = tyser_full.var.index[tyser_full.var["symbol"] == g].tolist()
    print(f"  {g:8s} {'FOUND ' + str(hit) if hit else 'ABSENT'}")

rawsample = (tyser_full.X.data[:200_000] if sp.issparse(tyser_full.X)
             else np.asarray(tyser_full.X[:200]).ravel())
looks_logged = not np.all(rawsample == np.round(rawsample))
print(f"\n.raw appears log-normalised: {looks_logged} (max {rawsample.max():.2f})")
if not looks_logged:
    print("  -> .raw holds counts; normalise before plotting:")
    print("     sc.pp.normalize_total(tyser_full, target_sum=1e4); sc.pp.log1p(tyser_full)")

print(f"\ncluster_id classes: {tyser_full.obs['cluster_id'].nunique()}")
print(f"obsm: {list(tyser_full.obsm.keys())}")

if WRITE and not (TYSER_OUT.exists() and not OVERWRITE):
    ad.settings.allow_write_nullable_strings = True
    tyser_full.write_h5ad(TYSER_OUT)
    print(f"\nWrote {TYSER_OUT.name} ({TYSER_OUT.stat().st_size/1e6:.0f} MB)")
else:
    print(f"\nNot written (WRITE={WRITE}, exists={TYSER_OUT.exists()})")

## Next steps

With `adata_master_normalized.h5ad` in place:

- **Figure 2 dot plots** — rerun your existing dot plot cells against the new object with `use_raw=False`. They will now show normalised expression rather than raw UMI counts.
- **MAGIC** — run it **per site** (Michigan and Chicago separately), following your collaborator's cell 209 pattern, and write into `layers['magic']`. Imputing across sites would smear replicate structure into the gene signal.
- **Grouping** — use `timepoint_true` and `condition_time_true` for all figure labels so nothing says "48h".